# Automated KPI-based candidate selection

This notebook demonstrates automated candidate selection using experimentally reported
flow-battery key performance indicators (KPI) retrieved through the OFBO virtual
knowledge graph.

The workflow is configured for identifying aqueous negolyte candidates. It evaluates:

- redox potential, \(E^0\), where lower values are preferred for negolyte operation;
- diffusion coefficient, \(D\), where higher values support mass transport;
- heterogeneous electron-transfer rate constant, \(k^0\), where higher values indicate
  faster charge-transfer kinetics;
- peak potential gap, \(\Delta E_\mathrm{p}\), where lower values indicate more
  reversible electrochemistry; and
- solubility, \(s\), where higher values enable greater active-material concentration.

The screening criteria are explicitly defined in Python. This makes the procedure
transparent, reproducible, and easy to adapt for different flow-battery design targets.

Missing values are retained rather than interpreted as poor performance. In particular,
a candidate with favourable electrochemical KPI but no reported solubility is identified
as a potential target for follow-up solubility measurement.

In [ ]:
from pathlib import Path
from io import StringIO
import numpy as np

import pandas as pd
import requests
from IPython.display import Markdown, display


ONTOP_ENDPOINT = "http://localhost:8080/sparql"
TIMEOUT_SECONDS = 120

## 2. SPARQL query

The query retrieves experimentally generated values for five KPI-relevant OFBO
property classes:

- redox potential;
- diffusion coefficient;
- heterogeneous electron-transfer rate constant;
- peak potential gap; and
- solubility.

Each result includes the molecular InChIKey and SMILES representation, the measured
value and unit, the chemical system, pH where available, and the primary literature
source.

The results are returned in a long format, with one row per property observation.
This avoids combining measurements from different experimental conditions during
SPARQL execution. The properties are subsequently grouped into candidate records in
Python.

Only redox potentials reported versus SHE are included, ensuring that the potential
values used in the screening are directly comparable.

In [2]:
query_file = Path("queries/query_kpi_candidates.rq")
query = query_file.read_text(encoding="utf-8")

display(Markdown(f"```sparql\n{query}\n```"))

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?inchiKey
    ?smiles
    ?propertyType
    ?value
    ?unit
    ?source
WHERE {
    ?molecule a ofbo:OFBO_0000012 ;
              schema:inChIKey ?inchiKey ;
              ofbo:OFBO_0009101 ?smiles ;
              ofbo:OFBO_0009206 ?reaction .

    ?reaction sio:SIO_000223 ?property .

    ?property a ?propertyType ;
              qudt:numericValue ?value ;
              qudt:unit ?unit ;
              prov:wasGeneratedBy ?experiment ;
              prov:hadPrimarySource ?source .

    ?experiment a ofbo:OFBO_0008500 .

    VALUES ?propertyType {
        ofbo:OFBO_0001104
        ofbo:OFBO_0001103
        ofbo:OFBO_0001402
        ofbo:OFBO_0001501
    }

    # Hydroxyphenazine dataset used for the automated-selection case study.
    FILTER(?source = <https://doi.org/10.1039/D1TA03655F>)
}
ORDER BY ?inchiKey ?propertyType
```

In [3]:
response = requests.post(
    ONTOP_ENDPOINT,
    data={"query": query},
    headers={"Accept": "text/csv"},
    timeout=TIMEOUT_SECONDS,
)

response.raise_for_status()

kpi_raw = pd.read_csv(StringIO(response.text)).drop_duplicates()

kpi_raw["value"] = pd.to_numeric(kpi_raw["value"], errors="coerce")

if "pH" in kpi_raw.columns:
    kpi_raw["pH"] = pd.to_numeric(kpi_raw["pH"], errors="coerce")

kpi_raw = kpi_raw.dropna(
    subset=["inchiKey", "propertyType", "value"]
).copy()

print(f"Retrieved {len(kpi_raw)} KPI observations.")
print(f"Retrieved {kpi_raw['inchiKey'].nunique()} unique molecules.")

kpi_raw.head()

Retrieved 27 KPI observations.
Retrieved 7 unique molecules.


,inchiKey,smiles,propertyType,value,unit,source
0,FSNHVRFXRAUXFJ-UHFFFAOYSA-N,c1ccc2nc3ccccc3nc2c1,https://w3id.org/ofbo#OFBO_0001103,530.000000,http://qudt.org/vocab/unit/MilliV,https://doi.org/10.1039/D1TA03655F
1,FSNHVRFXRAUXFJ-UHFFFAOYSA-N,c1ccc2nc3ccccc3nc2c1,https://w3id.org/ofbo#OFBO_0001104,-0.740000,http://qudt.org/vocab/unit/V,https://doi.org/10.1039/D1TA03655F
2,FSNHVRFXRAUXFJ-UHFFFAOYSA-N,c1ccc2nc3ccccc3nc2c1,https://w3id.org/ofbo#OFBO_0001402,0.000360,http://qudt.org/vocab/unit/CentiM-PER-SEC,https://doi.org/10.1039/D1TA03655F
3,FSNHVRFXRAUXFJ-UHFFFAOYSA-N,c1ccc2nc3ccccc3nc2c1,https://w3id.org/ofbo#OFBO_0001501,0.000001,http://qudt.org/vocab/unit/CentiM2-PER-SEC,https://doi.org/10.1039/D1TA03655F
4,GDKLWFNUGWTYRJ-UHFFFAOYSA-N,c1ccc2nc3ccccc3nc2c1,https://w3id.org/ofbo#OFBO_0001103,561.000000,http://qudt.org/vocab/unit/MilliV,https://doi.org/10.1039/D1TA03655F


## 3. Multi-criterion KPI-based candidate analysis

The candidate-selection workflow applies four explicit electrochemical criteria to the hydroxyphenazine dataset. The criteria represent favourable redox potential for negolyte operation, sufficiently rapid diffusion, sufficiently rapid heterogeneous electron transfer, and electrochemical reversibility.

For this demonstration, candidates are required to have:

- a redox potential at or below −0.50 V;
- a diffusion coefficient at or above 1 × 10⁻⁵ cm² s⁻¹;
- a heterogeneous electron-transfer rate constant at or above 1 × 10⁻³ cm s⁻¹; and
- a peak potential gap at or below 100 mV.

The table reports the number of criteria satisfied by each molecule and records the reason for deprioritisation. This makes the automated decision process interpretable rather than relying on a single ranking metric.

In [4]:
TYPE_REDOX = "https://w3id.org/ofbo#OFBO_0001104"
TYPE_DELTA_EP = "https://w3id.org/ofbo#OFBO_0001103"
TYPE_K0 = "https://w3id.org/ofbo#OFBO_0001402"
TYPE_DIFFUSION = "https://w3id.org/ofbo#OFBO_0001501"

property_names = {
    TYPE_REDOX: "E0_V",
    TYPE_DELTA_EP: "DeltaEp_mV",
    TYPE_K0: "k0_cm_s",
    TYPE_DIFFUSION: "D_cm2_s",
}

kpi = kpi_raw.copy()
kpi["property"] = kpi["propertyType"].map(property_names)

candidates = (
    kpi.pivot_table(
        index=["inchiKey", "smiles"],
        columns="property",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)

# Screening thresholds for the negolyte demonstration.
MAX_REDOX_POTENTIAL = -0.50
MIN_DIFFUSION = 1e-5
MIN_RATE_CONSTANT = 1e-3
MAX_PEAK_GAP = 100

candidates["redoxPass"] = (
    candidates["E0_V"] <= MAX_REDOX_POTENTIAL
)

candidates["diffusionPass"] = (
    candidates["D_cm2_s"] >= MIN_DIFFUSION
)

candidates["kineticsPass"] = (
    candidates["k0_cm_s"] >= MIN_RATE_CONSTANT
)

candidates["reversibilityPass"] = (
    candidates["DeltaEp_mV"] <= MAX_PEAK_GAP
)

criteria_columns = [
    "redoxPass",
    "diffusionPass",
    "kineticsPass",
    "reversibilityPass",
]

candidates["criteriaPassed"] = candidates[
    criteria_columns
].sum(axis=1)

candidates["candidateStatus"] = np.where(
    candidates[criteria_columns].all(axis=1),
    "Prioritised for solubility measurement",
    "Deprioritised",
)


def screening_reason(row):
    """Provide a concise explanation for candidates that fail screening."""
    reasons = []

    if pd.isna(row["E0_V"]):
        reasons.append("missing redox potential")
    elif not row["redoxPass"]:
        reasons.append("unfavourable redox potential")

    if not row["diffusionPass"]:
        reasons.append("low diffusion")

    if not row["kineticsPass"]:
        reasons.append("slow electron transfer")

    if not row["reversibilityPass"]:
        reasons.append("large peak gap")

    return "; ".join(reasons) if reasons else "all KPI criteria satisfied"


candidates["screeningReason"] = candidates.apply(
    screening_reason,
    axis=1,
)

ranked_candidates = candidates.sort_values(
    by=["criteriaPassed", "D_cm2_s"],
    ascending=[False, False],
).reset_index(drop=True)

candidate_table = ranked_candidates[
    [
        "inchiKey",
        "D_cm2_s",
        "E0_V",
        "k0_cm_s",
        "DeltaEp_mV",
        "criteriaPassed",
        "candidateStatus",
        "screeningReason",
    ]
]

candidate_table

property,inchiKey,D_cm2_s,E0_V,k0_cm_s,DeltaEp_mV,criteriaPassed,candidateStatus,screeningReason
0,UVPLITJCYDVORW-UHFFFAOYSA-N,0.000017,-1.00,0.00386,66.0,4,Prioritised for solubility measurement,all KPI criteria satisfied
1,JOXNFMAXWAPITK-UHFFFAOYSA-N,0.000021,-0.84,0.00633,673.0,3,Deprioritised,large peak gap
2,YYGYKQIAGNVTNM-UHFFFAOYSA-N,0.000012,-0.95,0.00102,526.0,3,Deprioritised,large peak gap
3,GDKLWFNUGWTYRJ-UHFFFAOYSA-N,0.000014,-0.90,0.00048,561.0,2,Deprioritised,slow electron transfer; large peak gap
4,VAASABGZPGHBSZ-UHFFFAOYSA-N,0.000005,-0.66,0.00108,484.0,2,Deprioritised,low diffusion; large peak gap
5,RSGCOGXEEDICHO-UHFFFAOYSA-N,0.000016,NaN,0.00068,348.0,1,Deprioritised,missing redox potential; slow electron transfe...
6,FSNHVRFXRAUXFJ-UHFFFAOYSA-N,0.000001,-0.74,0.00036,530.0,1,Deprioritised,low diffusion; slow electron transfer; large p...
